In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [3]:
import pandas as pd

if IN_COLAB:
    df = pd.read_csv("hf://datasets/ahmedheakl/resume-atlas/train.csv")
else:
    df = pd.read_csv('data/raw/train.csv')

print("=" * 60)
print("INITIAL DATASET INFO")
print("=" * 60)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

INITIAL DATASET INFO
Dataset shape: (13389, 2)
Columns: ['Category', 'Text']
Memory usage: 51.27 MB


In [4]:
# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
Category    0
Text        0
dtype: int64


In [5]:
# Check data types
print("\nData types:")
print(df.dtypes)

print("\n" + "=" * 60)
print("CATEGORY ANALYSIS")
print("=" * 60)


Data types:
Category    str
Text        str
dtype: object

CATEGORY ANALYSIS


In [6]:
df.sample(10)

,Category,Text
13253,Web Designing,professional summary jessica claire 100 montgo...
11959,Public Relations,jessica claire 100 montgomery st 10th floor 55...
2729,Accountant,ethan brown 1 main street new cityland ca 9101...
5117,Mechanical Engineer,john smith eit 22 street rd city state 00000 m...
8376,Data Science,summary jessica claire montgomery street san f...
2457,SAP Developer,isabelle todd experienced sap abap developer l...
9718,Electrical Engineering,jessica claire montgomery street san francisco...
10003,Finance,professional summary skills jessica claire 100...
146,Advocate,client advocate robert smith phone 123 456 78 ...
1306,Education,rachel schuli elementary school teacher patien...


In [7]:
# Analyze categories
category_counts = df['Category'].value_counts()
print(f"Number of unique categories: {len(category_counts)}")
print("\nTop 10 categories:")
print(category_counts.head(10))


Number of unique categories: 43

Top 10 categories:
Category
Education                 410
Electrical Engineering    384
Mechanical Engineer       384
Consultant                368
Civil Engineer            364
Sales                     364
Management                361
Human Resources           360
Digital Media             358
Accountant                350
Name: count, dtype: int64


In [8]:
# Check for any inconsistencies in category names
print("\nAll unique categories:")
for i, cat in enumerate(sorted(df['Category'].unique()), 1):
    print(f"{i:2d}. {cat}")

print("\n" + "=" * 60)
print("TEXT DATA ANALYSIS")
print("=" * 60)


All unique categories:
 1. Accountant
 2. Advocate
 3. Agriculture
 4. Apparel
 5. Architecture
 6. Arts
 7. Automobile
 8. Aviation
 9. BPO
10. Banking
11. Blockchain
12. Building and Construction
13. Business Analyst
14. Civil Engineer
15. Consultant
16. Data Science
17. Database
18. Designing
19. DevOps
20. Digital Media
21. DotNet Developer
22. ETL Developer
23. Education
24. Electrical Engineering
25. Finance
26. Food and Beverages
27. Health and Fitness
28. Human Resources
29. Information Technology
30. Java Developer
31. Management
32. Mechanical Engineer
33. Network Security Engineer
34. Operations Manager
35. PMO
36. Public Relations
37. Python Developer
38. React Developer
39. SAP Developer
40. SQL Developer
41. Sales
42. Testing
43. Web Designing

TEXT DATA ANALYSIS


In [9]:
# Analyze text data
print("Sample text before cleaning:")
print(df['Text'].iloc[0][:800] + "...")

Sample text before cleaning:
education omba executive leadership university texas 20162018 bachelor science accounting richland college 20052008 training certifications certified management accountant cma certified financial modeling valuation analyst compliance antimoney laundering 092016 american institute banking certified public account cpa lean six sigma green belt certified trade products financial regulations 082016 american institute banking achievements speaker bringing leader within 082019 successfully presented empowering speech leadership 500 participants speaker dallas convention cpas 032019 successfully delivered seminar 3k cpas convention guests teaching experience online teacher udemy 2017 taught online accounting nonaccountant course udemy similar online teaching platforms developed effective teaching...


In [10]:
# Check text lengths
text_lengths = df['Text'].str.len()
print(f"\nText length statistics:")
print(f"Mean: {text_lengths.mean():.0f} characters")
print(f"Median: {text_lengths.median():.0f} characters")
print(f"Min: {text_lengths.min():.0f} characters")
print(f"Max: {text_lengths.max():.0f} characters")


Text length statistics:
Mean: 3987 characters
Median: 3367 characters
Min: 44 characters
Max: 56216 characters


In [11]:
#defining cleaning
import re
def clean_text(text):
    """
    Comprehensive text cleaning function for resume data
    """
    if pd.isna(text):
        return ""

    # Convert to string if not already
    text = str(text)

    # Remove BOM and other invisible characters
    text = text.replace('\ufeff', '')  # BOM character
    text = text.replace('﻿', '')        # Another BOM variant

    # Handle various line breaks and whitespace
    text = re.sub(r'\r\n', ' ', text)   # Windows line breaks
    text = re.sub(r'\r', ' ', text)     # Mac line breaks
    text = re.sub(r'\n', ' ', text)     # Unix line breaks
    text = re.sub(r'\t', ' ', text)     # Tabs

    # Remove multiple underscores (common in resume templates)
    text = re.sub(r'_{3,}', ' ', text)

    # Remove email addresses (for privacy)
    text = re.sub(r'\S+@\S+', 'EMAIL', text)

    # Remove phone numbers (basic pattern)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', 'PHONE', text)
    text = re.sub(r'\(\d{3}\)\s?\d{3}[-.]?\d{4}', 'PHONE', text)

    # Remove URLs
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', 'URL', text)
    text = re.sub(r'www\.(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', 'URL', text)

    # Remove excessive punctuation
    text = re.sub(r'[•▪▫◦‣⁃]', ' ', text)  # Bullet points
    text = re.sub(r'[^\w\s\-.,;:()&+/%]', ' ', text)  # Keep basic punctuation

    # Clean up multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Strip whitespace
    text = text.strip()

    return text

def clean_category(category):
    """
    Clean category names for consistency
    """
    if pd.isna(category):
        return "Unknown"

    category = str(category).strip()

    # Standardize common variations
    category_mapping = {
        'Web Designing': 'Web Design',
        'Information Technology': 'IT',
        'Electrical Engineering': 'Electrical Engineer',
        'Business Analyst': 'Business Analysis',
        'React Developer': 'React Development',
        'ETL Developer': 'ETL Development',
        'SAP Developer': 'SAP Development',
        'DevOps': 'DevOps Engineer',
        'Database': 'Database Administration',
        'Designing': 'Design'
    }

    return category_mapping.get(category, category)

In [12]:
# Apply cleaning functions
print("Cleaning text data...")
df['Text_Clean'] = df['Text'].apply(clean_text)

print("Cleaning category data...")
df['Category_Clean'] = df['Category'].apply(clean_category)

Cleaning text data...
Cleaning category data...


In [13]:
# Remove too-short texts
min_text_length = 50
initial_count = len(df)
df = df[df['Text_Clean'].str.len() >= min_text_length]
print(f"Removed {initial_count - len(df)} rows with text shorter than {min_text_length} characters")

# Remove duplicates
initial_count = len(df)
df = df.drop_duplicates(subset=['Text_Clean'], keep='first')
print(f"Removed {initial_count - len(df)} duplicate rows")


Removed 1 rows with text shorter than 50 characters
Removed 1304 duplicate rows


In [14]:
# Analyze cleaned categories
category_counts_clean = df['Category_Clean'].value_counts()
print(f"\nNumber of unique categories after cleaning: {len(category_counts_clean)}")
print("\nTop 10 categories after cleaning:")
print(category_counts_clean.head(10))


Number of unique categories after cleaning: 43

Top 10 categories after cleaning:
Category_Clean
Education                    388
Electrical Engineer          360
Consultant                   344
Sales                        342
Digital Media                340
Accountant                   337
Building and Construction    335
Mechanical Engineer          335
Finance                      331
Aviation                     327
Name: count, dtype: int64


In [15]:
# Sample cleaned text
print("\nSample cleaned text:")
print(df['Text_Clean'].iloc[0][:500] + "...")


Sample cleaned text:
education omba executive leadership university texas 20162018 bachelor science accounting richland college 20052008 training certifications certified management accountant cma certified financial modeling valuation analyst compliance antimoney laundering 092016 american institute banking certified public account cpa lean six sigma green belt certified trade products financial regulations 082016 american institute banking achievements speaker bringing leader within 082019 successfully presented e...


In [16]:
# Check for any remaining issues
print("\nChecking for remaining issues...")
print(f"Empty texts: {(df['Text_Clean'].str.len() == 0).sum()}")
print(f"Texts with only spaces: {(df['Text_Clean'].str.strip().str.len() == 0).sum()}")


Checking for remaining issues...
Empty texts: 0
Texts with only spaces: 0


In [17]:
# Save cleaned data
df_final = df[['Category_Clean', 'Text_Clean']].copy()
df_final.columns = ['Category', 'Text']

In [18]:
# Save as both parquet and CSV for flexibility
import os
os.makedirs('data/processed', exist_ok=True)
df_final.to_parquet('data/processed/resume_data_cleaned.parquet', index=False)
df_final.to_csv('data/processed/resume_data_cleaned.csv', index=False)

In [19]:
# Display final sample
print("\nFinal cleaned dataset sample:")
print(df_final.head())

print("\nCLEANING COMPLETE!")


Final cleaned dataset sample:
     Category                                               Text
0  Accountant  education omba executive leadership university...
1  Accountant  howard gerrard accountant deyjobcom birmingham...
2  Accountant  kevin frank senior accountant inforesumekraftc...
3  Accountant  place birth nationality olivia ogilvy accounta...
4  Accountant  stephen greet cpa senior accountant 9 year exp...

CLEANING COMPLETE!


In [20]:
# print("Job categories and counts:")
# print(df['Category'].value_counts())

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [22]:
df = pd.read_csv('/content/resume_data_cleaned.csv' if IN_COLAB else 'data/processed/resume_data_cleaned.csv')  # path as needed
print("load successful")


load successful


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    df['Text'], df['Category'], test_size=0.2, random_state=42, stratify=df['Category']
)


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)
